# 02C | CrisisMMD: data understanding, preparation and EDA

Reconcile text annotations, quarantine conflicts and inspect screening/humanitarian classes by event. Compare minimal cleaning and URL/mention normalization on matched shared-task cohorts. Preserve rare classes and held-out membership; explain exclusions and token-length differences.

No predictive models are trained here. This notebook processes only CrisisMMD.


## Inputs and outputs

Complete notebook 02B. Inputs are this experiment's public acquisition manifests and raw CrisisMMD files, restored from Google Drive.

Outputs: Eight panels per variant (sixteen total), matched cohort evidence .


## Configuration

Enable Drive, bootstrap and stage execution. Keep the experiment ID unchanged across the package. All actions default disabled.


In [ ]:
EXPERIMENT_ID = "colab-cpu-v3-clean"
DRIVE_FOLDER = "ChainShield-Colab"
ENABLE_DRIVE = False
BOOTSTRAP_EXPERIMENT = False
ALLOW_PACKAGE_INSTALL = False
ALLOW_DATA_DOWNLOADS = False
ALLOW_PRETRAINED_DOWNLOADS = False
RUN_STAGE = False
SYNC_INTERVAL_SECONDS = 60
VERIFY_PERSISTENCE_FULL = False
VERIFY_CHECKPOINT_PAIRS = False
CLEAR_STALE_WRITER_LOCK = False
CONFIRM_NO_OTHER_WRITER = False
DOWNLOAD_REPORT_ZIP = False

# Isolated plotting workers must not inherit the Colab inline backend.
if RUN_STAGE:
    import os
    os.environ["MPLBACKEND"] = "Agg"


## Shared definitions

Readable code is supplied once by its owning stage and persisted in Drive. Register these definitions before attaching; registration alone performs no preparation.


In [ ]:
# This small cell registers the visible stage code; it writes no files.
REGISTERED_FILES = {}
def register_stage_file(line, body):
    name=line.strip()
    if not name or name.startswith("/") or "\\" in name or ":" in name or ".." in name.split("/"):
        raise ValueError("Use a safe relative module name")
    REGISTERED_FILES[name]=body.rstrip("\n")+"\n"
get_ipython().register_magic_function(register_stage_file,"cell","chainshield_file")


### src/chainshield/data/crisismmd.py


In [ ]:
%%chainshield_file src/chainshield/data/crisismmd.py
"""Reconcile CrisisMMD text-specific annotations and preserve split provenance."""
from __future__ import annotations
from collections import Counter, defaultdict
import csv
import json
from pathlib import Path
from .text_utils import clean_text, text_key

SPLITS = ("train","dev","test")


def resolve_votes(values, allowed):
    votes = Counter(str(v).strip() for v in values
                    if v is not None and str(v).strip().casefold() not in {"","none","nan","null","n/a"})
    if not votes:
        return "", "missing", votes
    if any(v not in allowed for v in votes):
        return "", "invalid", votes
    if len(votes) != 1:
        return "", "conflict", votes
    return next(iter(votes)), "unanimous", votes


def shared_split(splits):
    known = [s for s in splits if s in SPLITS]
    return max(known, key=SPLITS.index) if known else ""


def apply_text_policy(canonical, task, view):
    groups = defaultdict(list)
    result = []
    for row in canonical:
        split = row[task+"_split"] if view == "original_task" else row["shared_split"]
        reasons = []
        if split not in SPLITS:
            reasons.append("missing_or_conflicting_task_split")
        # Shared assignment never makes an absent original task split eligible.
        if row[task+"_split"] not in SPLITS:
            reasons.append("missing_or_conflicting_original_task_split")
        if row["text_status"] not in {"unanimous","normalized_agreement"}:
            reasons.append("text_"+row["text_status"])
        if not row["text_normalized"]:
            reasons.append("empty_text")
        if row[task+"_status"] != "unanimous":
            reasons.append("tsv_label_"+row[task+"_status"])
        if row["event_count"] != 1:
            reasons.append("ambiguous_event")
        entry = {
            "record_id":row["tweet_id"],"task":task,"view":view,
            "group_id":row["text_key"] or "empty:"+row["tweet_id"],
            "text_key":row["text_key"],"label":row[task+"_label"],
            "split":split,"_reasons":reasons,
        }
        result.append(entry)
        groups[entry["group_id"]].append(entry)
    for group in groups.values():
        splits = {r["split"] for r in group if r["split"] in SPLITS}
        labels = {r["label"] for r in group if r["label"]}
        candidates = sorted((r for r in group if not r["_reasons"]),
                            key=lambda r:r["record_id"])
        representative = candidates[0]["record_id"] if candidates else None
        for row in group:
            if len(splits)>1:
                row["_reasons"].append("cross_split_exact_text")
            if len(labels)>1:
                row["_reasons"].append("same_text_conflicting_labels")
            if (not row["_reasons"] and row["record_id"] != representative):
                row["_reasons"].append("redundant_same_split_text")
            row["eligible"] = not row["_reasons"]
            row["exclusion_reasons"] = "|".join(sorted(set(row.pop("_reasons"))))
    return result


def prepare(raw_root: Path, config):
    spec = config["crisismmd"]
    annotations, json_rows = defaultdict(list), defaultdict(list)
    for path in sorted((raw_root/"files_individual_events").glob("*_final_data.tsv")):
        with path.open(encoding="utf-8-sig",newline="") as handle:
            reader = csv.DictReader(handle,delimiter="\t")
            required = {"tweet_id","tweet_text",*spec["annotation_fields"].values()}
            if not required.issubset(reader.fieldnames or []):
                raise ValueError(f"Missing TSV fields: {path.name}")
            for row in reader:
                if None in row:
                    raise ValueError(f"Malformed annotation row: {path.name}")
                row = dict(row,source_file=path.relative_to(raw_root).as_posix(),
                           event_name=path.stem.removesuffix("_final_data"))
                annotations[str(row["tweet_id"])].append(row)
    if not annotations:
        raise ValueError("No CrisisMMD text annotations found")
    for task in spec["labels"]:
        for split in SPLITS:
            path = raw_root/task/(split+".json")
            data = json.loads(path.read_text(encoding="utf-8-sig"))
            if not isinstance(data,list):
                raise ValueError("Expected JSON record arrays")
            for row in data:
                if not {"tweet_id","tweet_text","label","event_name"}.issubset(row):
                    raise ValueError("Missing JSON fields")
                enriched = dict(row,task=task,split=split,
                                source_file=path.relative_to(raw_root).as_posix())
                json_rows[str(row["tweet_id"])].append(enriched)
    canonical, provenance = [], []
    for tid in sorted(set(json_rows)|set(annotations)):
        js, ann = json_rows[tid], annotations[tid]
        variants = [clean_text(r.get("tweet_text")) for r in js+ann if clean_text(r.get("tweet_text"))]
        normalized_keys = {text_key(s) for s in variants}
        status = ("missing" if not variants else "conflict" if len(normalized_keys)>1
                  else "unanimous" if len(set(variants))==1 else "normalized_agreement")
        text = variants[0] if variants else ""
        events = sorted({r["event_name"] for r in js+ann if r.get("event_name")})
        row = {
            "tweet_id":tid,"event_name":"|".join(events),"event_count":len(events),
            "text_original":next((r.get("tweet_text","") for r in js+ann
                                  if r.get("tweet_text")),""),
            "text_normalized":text,"text_key":text_key(text),"text_status":status,
            "json_annotation_rows":len(js),"tsv_annotation_rows":len(ann),
        }
        audit = {
            "tweet_id":tid,"json_source_files":"|".join(sorted({r["source_file"] for r in js})),
            "tsv_source_files":"|".join(sorted({r["source_file"] for r in ann})),
            "text_status":status,"normalized_text_variants":len(normalized_keys),
        }
        for task, allowed in spec["labels"].items():
            original_splits = sorted({r["split"] for r in js if r["task"]==task})
            label, state, votes = resolve_votes(
                [r.get(spec["annotation_fields"][task]) for r in ann], allowed)
            json_votes = Counter(str(r["label"]) for r in js if r["task"]==task)
            row.update({
                task+"_label":label,task+"_status":state,
                task+"_split":original_splits[0] if len(original_splits)==1 else "",
                task+"_json_disagreement":bool(label and any(v!=label for v in json_votes)),
            })
            audit.update({
                task+"_original_splits":"|".join(original_splits),
                task+"_tsv_votes":json.dumps(dict(votes),sort_keys=True),
                task+"_json_votes":json.dumps(dict(json_votes),sort_keys=True),
            })
        row["shared_split"] = shared_split([r["split"] for r in js])
        canonical.append(row)
        provenance.append(audit)
    manifest = []
    for task in spec["labels"]:
        for view in spec["views"]:
            entries = apply_text_policy(canonical,task,view)
            for row,entry in zip(canonical,entries):
                row[f"{task}_{view}_eligible"] = entry["eligible"]
                row[f"{task}_{view}_exclusion_reasons"] = entry["exclusion_reasons"]
            manifest.extend(entries)
    quality = {
        "canonical_tweets":len(canonical),"unique_json_tweets":len(json_rows),
        "tsv_only_tweets":len(set(annotations)-set(json_rows)),
        "json_annotation_rows":sum(map(len,json_rows.values())),
        "tsv_annotation_rows":sum(map(len,annotations.values())),
        "cross_task_split_differences":sum(r["informative_split"]!=r["humanitarian_split"]
                                           for r in canonical),
        "text_status_counts":dict(Counter(r["text_status"] for r in canonical)),
        "shared_base_split_counts":dict(Counter(r["shared_split"] for r in canonical)),
        "tasks":{},
        "limitations":[
            "TSV precedence is an explicit annotation policy, not proof of superiority to JSON labels.",
            "Eligibility is task-specific; an unresolved second task does not discard a valid first task.",
            "Original-task and shared views are separate experiments, not interchangeable benchmarks.",
            "Use shared views for cascaded evaluation and select all models using that view consistently.",
            "Near-duplicate and event-held-out evaluation remain pending.",
            "Damage severity and factual verification are unsupported text-only targets.",
            "Some minority classes may be too small for reliable performance claims.",
        ],
    }
    for task in spec["labels"]:
        task_report = {
            "text_label_counts":dict(Counter(r[task+"_label"] or "UNRESOLVED" for r in canonical)),
            "annotation_status_counts":dict(Counter(r[task+"_status"] for r in canonical)),
            "json_disagreement_tweets":sum(r[task+"_json_disagreement"] for r in canonical),
            "views":{},
        }
        for view in spec["views"]:
            entries = [r for r in manifest if r["task"]==task and r["view"]==view]
            eligible = [r for r in entries if r["eligible"]]
            task_report["views"][view] = {
                "eligible_rows":len(eligible),
                "split_counts":dict(Counter(r["split"] for r in eligible)),
                "split_label_counts":{s:dict(Counter(r["label"] for r in eligible if r["split"]==s))
                                      for s in SPLITS},
                "exclusion_reason_counts":dict(Counter(reason for r in entries
                    for reason in r["exclusion_reasons"].split("|") if reason)),
            }
        quality["tasks"][task] = task_report
    return {"canonical":canonical,"split_manifest":manifest,"provenance":provenance,
            "quality":quality,"label_map":spec["labels"]}


### configs/preprocessing/crisismmd.json


In [ ]:
%%chainshield_file configs/preprocessing/crisismmd.json
{
  "schema_version": 1,
  "pipeline_version": "chainshield-preparation-v1",
  "seed": 42,
  "splits": {
    "train": 0.7,
    "dev": 0.15,
    "test": 0.15
  },
  "normalization": "HTML unescape, Unicode NFC, whitespace collapse; casefold only for exact-text keys",
  "crisismmd": {
    "label_policy": "Unanimous nonempty text-specific TSV votes; retain JSON votes for audit; conflicts/missing/invalid labels quarantined per task",
    "split_policy": "Preserve original per-task views; shared view assigns latest held-out partition across tasks; quarantine exact text spanning partitions",
    "labels": {
      "informative": [
        "informative",
        "not_informative"
      ],
      "humanitarian": [
        "affected_individuals",
        "infrastructure_and_utility_damage",
        "injured_or_dead_people",
        "missing_or_found_people",
        "not_humanitarian",
        "other_relevant_information",
        "rescue_volunteering_or_donation_effort",
        "vehicle_damage"
      ]
    },
    "annotation_fields": {
      "informative": "text_info",
      "humanitarian": "text_human"
    },
    "views": [
      "original_task",
      "shared"
    ],
    "exclude": [
      "images",
      "image_damage",
      "image_info",
      "image_human"
    ]
  }
}


### src/chainshield/research/preparation_crisismmd.py


In [ ]:
%%chainshield_file src/chainshield/research/preparation_crisismmd.py
from collections import defaultdict,Counter
from copy import deepcopy
import json,re
from .preparation import sha,yes,clean,normalize_urls_mentions,key,csv_rows,canonical_scalar,records_digest,manifest_key,frozen_test_digest,exclude,validate_partition_groups,write_csv


def comparison_cohort(canonical,manifest):
    """Use normalized text only for structural isolation; inspect labels in training only."""
    by_id={r["tweet_id"]:r for r in canonical}
    result=deepcopy(manifest)
    groups=defaultdict(list)
    for row in result:
        normalized_key=key(normalize_urls_mentions(by_id[row["record_id"]]["text_normalized"]))
        row["comparison_text_key"]=normalized_key
        row["group_id"]=normalized_key
        view=row.get("view","")
        task_scope=row["task"] if view=="original_task" else "shared_tasks"
        groups[(view,task_scope,normalized_key)].append(row)
    priority={"train":0,"dev":1,"test":2}
    for members in groups.values():
        eligible=[r for r in members if yes(r["eligible"])]
        if not eligible:continue
        latest=max(priority[r["split"]] for r in eligible)
        for row in eligible:
            if priority[row["split"]]<latest:
                exclude(row,"research_normalization_cross_partition")
    train_groups=defaultdict(list)
    for row in result:
        if row["split"]=="train" and yes(row["eligible"]):
            train_groups[(row.get("view",""),row["task"],row["group_id"])].append(row)
    for group in train_groups.values():
        if len({r["label"] for r in group})>1:
            for row in group:exclude(row,"research_normalization_training_label_conflict")
        else:
            representative=min(r["record_id"] for r in group)
            for row in group:
                if row["record_id"]!=representative:exclude(row,"research_normalization_training_duplicate")
    if frozen_test_digest(result)!=frozen_test_digest(manifest):
        raise ValueError("Final-test identities or labels changed")
    return result

def apply_variant(base,cohort,normalized):
    bundle=deepcopy(base)
    bundle["split_manifest"]=deepcopy(cohort)
    by_id={r["tweet_id"]:r for r in bundle["canonical"]}
    for row in bundle["canonical"]:
        original_key=row["text_key"]
        row["research_original_text_key"]=original_key
        row["research_normalization_policy"]="url_mentions" if normalized else "minimal_control"
        if normalized:
            row["text_normalized"]=normalize_urls_mentions(row["text_normalized"])
            row["text_key"]=key(row["text_normalized"])
    for row in bundle["split_manifest"]:
        canonical=by_id[row["record_id"]]
        row["text_key"]=canonical["text_key"]
        prefix=row["task"]+"_"+row["view"]
        canonical[prefix+"_eligible"]=yes(row["eligible"])
        canonical[prefix+"_exclusion_reasons"]=row["exclusion_reasons"]
    validate_partition_groups(bundle["split_manifest"])
    return bundle


### src/chainshield/research/eda_crisismmd.py


In [ ]:
%%chainshield_file src/chainshield/research/eda_crisismmd.py
import json
from .eda import truth,text_profile,rows,digest

def add_profiles(profile,canonical,prepared,tokenizer,include_examples,root,inherited,config):
    source=profile["source_sha256"]
    crisis={r["tweet_id"]:r for r in canonical.get("crisismmd",[])}
    manifest=rows(prepared/"crisismmd/split_manifest.csv") if "crisismmd" in canonical else []
    vocab_path=root/"artifacts/neural/main/screening/main/vocabulary.json"
    vocabulary=json.loads(vocab_path.read_text()) if inherited and vocab_path.exists() else None
    if vocabulary is not None:source[vocab_path.relative_to(root).as_posix()]=digest(vocab_path)
    for task in (("informative","humanitarian") if "crisismmd" in canonical else ()):
        entries=[r for r in manifest if r["task"]==task and r["view"]=="shared" and r["split"]=="train" and truth(r["eligible"])]
        if len({r["record_id"] for r in entries})!=len(entries):raise ValueError("Repeated shared-view training IDs")
        records=[{"record_id":r["record_id"],"group_id":r["group_id"],"split":"train","text":crisis[r["record_id"]]["text_normalized"],
                  "label":r["label"],"source":crisis[r["record_id"]]["event_name"]} for r in entries]
        profile["text_tasks"]["crisismmd/"+task]=text_profile(records,task,tokenizer,vocabulary if task=="informative" else None,include_examples)


In [ ]:
import hashlib,json
EXPECTED_STAGE_FILES={'src/chainshield/data/crisismmd.py': '92f377ebdc778b5dd91e7411672a79c125d4ab56b0d048b3451b2df6a1925d73', 'configs/preprocessing/crisismmd.json': '7bf077f16a18597a9e9c43c579b020e939f6842df443ead824f6e533d0c4e83c', 'src/chainshield/research/preparation_crisismmd.py': 'bbf3a3c6523d01a55a751a0df6e4894c4deed21e86ae7065d358a09315afe9f6', 'src/chainshield/research/eda_crisismmd.py': '2153df03945f33b3b2c0c5c42ae655b2fe71dcde17ae9506c5b0131c3bd3d324'}
assert set(REGISTERED_FILES)==set(EXPECTED_STAGE_FILES)
for name,expected in EXPECTED_STAGE_FILES.items():
    if hashlib.sha256(REGISTERED_FILES[name].encode()).hexdigest()!=expected:raise ValueError("Changed source: "+name)


## Attach and verify prerequisites


In [ ]:
# Connect the Colab experiment and load the shared runtime.
from pathlib import Path
CONTROLLER_SHA = '8c092c59baffc0804aeca86b4f0ac3fb6d4be3cd96eea96c9e7299e559d9c9e6'
IMPLEMENTATION_SHA256 = CONTROLLER_SHA
rt = None
if BOOTSTRAP_EXPERIMENT:
    if not ENABLE_DRIVE:
        raise PermissionError("Set ENABLE_DRIVE=True")
    from google.colab import drive
    drive.mount("/content/drive")
    runtime_path=Path("/content/drive/MyDrive")/DRIVE_FOLDER/"experiments"/EXPERIMENT_ID/"workspace/.replication/controller.py"
    if not runtime_path.is_file():
        raise RuntimeError("Complete Version 3 notebook 00 first; its shared controller is missing")
    runtime_code=runtime_path.read_text(encoding="utf-8")
    if hashlib.sha256(runtime_code.encode()).hexdigest()!=CONTROLLER_SHA:
        raise ValueError("Shared runtime version differs; do not mix notebook packages")
    exec(compile(runtime_code,"<Colab shared runtime>","exec"),globals())
    rt=Replica(EXPERIMENT_ID,drive_base=DRIVE_FOLDER,allow_drive=True)
    if CLEAR_STALE_WRITER_LOCK:
        rt.clear_stale_lock(acknowledge_no_other_writer=CONFIRM_NO_OTHER_WRITER)
else:
    print("Bootstrap disabled: no Drive access or filesystem changes")


In [ ]:
if RUN_STAGE:
    if rt is None:raise RuntimeError("Enable Drive/bootstrap first")
    rt.require_stage('01')
    rt.install_sources(REGISTERED_FILES)
    rt.environment(allow_install=ALLOW_PACKAGE_INSTALL)
else:
    print("Stage execution disabled")


## Data understanding and preparation choices

Reconcile text annotations, quarantine conflicts and inspect screening/humanitarian classes by event. Compare minimal cleaning and URL/mention normalization on matched shared-task cohorts. Preserve rare classes and held-out membership; explain exclusions and token-length differences.

Preparation verifies public input hashes and expected semantic/test-membership digests. All analysis below uses eligible training records; final-test labels never select settings.


In [ ]:
if RUN_STAGE:
    rt.execute("02C-prepare",["scripts/dataset_research_workflow.py","prepare","--dataset","crisismmd"],SYNC_INTERVAL_SECONDS)
else:
    print("Stage execution disabled")


## Pinned tokenizer for measured text lengths

Tokenizer assets are independent of classifier fine-tuning. India-Nepal acquires them once; CrisisMMD reuses the same verified assets. No model weights are downloaded here.


In [ ]:
if RUN_STAGE:
    path=rt.root/"artifacts/research-v2/tokenizer/distilbert/manifest.json"
    if not path.exists():
        if not ALLOW_PRETRAINED_DOWNLOADS:raise PermissionError("Enable pinned tokenizer acquisition with ALLOW_PRETRAINED_DOWNLOADS=True")
        rt.execute("02C-tokenizer",["scripts/acquire_tokenizer_assets.py"],SYNC_INTERVAL_SECONDS)
else:
    print("Stage execution disabled")


## Measured EDA, interpretation and decisions

Read the observation and preparation implication for every chart. Coverage and correlation do not prove a model will perform better.


In [ ]:
if RUN_STAGE:
    rt.execute("02C-eda",["scripts/dataset_research_workflow.py","eda","--dataset","crisismmd"],SYNC_INTERVAL_SECONDS)
else:
    print("Stage execution disabled")


In [ ]:
if RUN_STAGE:
    print(json.dumps(read_json(rt.root/"data/processed/"/"crisismmd"/"quality.json"),indent=2))
else:
    print("Stage execution disabled")


In [ ]:
if RUN_STAGE:
    from IPython.display import display,Image,Markdown
    for variant in ('control', 'url_mentions'):
        folder=rt.root/"artifacts/research-v2/eda"/("crisismmd_"+variant)
        display(Markdown("### CrisisMMD | "+variant))
        display(Markdown((folder/"EDA_Review.md").read_text()))
        for image in sorted((folder/"figures_v2").glob("*.png")):display(Image(filename=str(image)))
        summary=read_json(folder/"dataset-profiles.json")
        print("Preparation safeguards:",json.dumps(summary["preparation_controls"],indent=2))
else:
    print("Stage execution disabled")


## Persist and complete

Finish only after preparation and charts succeed. A restart of the Python kernel alone may retain partial scratch files; use a clean temporary Colab runtime for archive restoration when needed. Preserve Drive evidence.


In [ ]:
if RUN_STAGE:
    rt.finish_stage("02C")
else:
    print("Stage execution disabled")
